# فاحص بيئة بيان | Bayan Runtime Doctor

**إعداد وتقديم | Prepared and delivered by:** منال قيسي· Manal qaysi


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manal-qaysi/bayan-nlp-Manal-qaysi/blob/main/notebooks/00_runtime_doctor.ipynb)

شغّل الخلايا من الأعلى إلى الأسفل. نجاح CPU كافٍ؛ GPU ليس شرطًا.  
Run the cells top to bottom. CPU success is sufficient; a GPU is not required.


> 🟣 **نسخة مشروع منال قيسي (Manal-qaysi).** المختبر الأصلي من مادة الدورة، والخلايا المعنونة بـ«🟣 مشروعي» أُضيفت لمشروعي: تربط الدفتر بمستودعي وبياناته وتطبع نتائج قابلة للتتبع بصيغة `BAYAN_RESULT::…` يجمعها `scripts/collect_results.py`.
> Run with **Runtime → Restart session and run all**, then **File → Save a copy in GitHub** to `notebooks/00_runtime_doctor.ipynb` in `Manal-qaysi/bayan-nlp-Manal-qaysi`.

## ماذا سيفحص؟ | What it checks

- إصدار Python ونظام التشغيل.
- توفر المكتبات الأساسية الموجودة عادةً في Colab.
- سلامة UTF-8 والنص العربي.
- الجهاز الفعلي: CPU أو CUDA GPU.
- الذاكرة والمساحة المتاحة تقريبًا.
- الوصول إلى المواقع المطلوبة، بوصفه تحذيرًا لا شرط نجاح.
- إنشاء `runtime_report.json` بلا أسرار أو بيانات شخصية.


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import shutil
import sys
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

COURSE = "Bayan Applied NLP"
AUTHOR = "Manal Qaysi"  # course: Meaad Al-Marri

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"

def gib(value: int) -> float:
    return round(value / (1024 ** 3), 2)

try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    in_colab = False
torch_version = package_version("torch")
cuda_available = False
gpu_name = None

if torch_version != "not-installed":
    import torch
    cuda_available = bool(torch.cuda.is_available())
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)

disk = shutil.disk_usage("/")
report = {
    "course": COURSE,
    "author": AUTHOR,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "in_colab": in_colab,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": "cuda" if cuda_available else "cpu",
    "gpu_name": gpu_name,
    "disk_free_gib": gib(disk.free),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scikit-learn": package_version("scikit-learn"),
        "torch": torch_version,
    },
}

print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "course": "Bayan Applied NLP",
  "author": "Manal Qaysi",
  "checked_at_utc": "2026-09-30T13:36:05.410912+00:00",
  "in_colab": true,
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "device": "cpu",
  "gpu_name": null,
  "disk_free_gib": 87.23,
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "torch": "2.11.0+cpu"
  }
}


In [2]:
# Core checks: these determine readiness. Network and GPU do not.
arabic_sample = "مرحبًا بكم في مشروع بيان"
core_checks = {
    "python_3_10_or_newer": sys.version_info >= (3, 10),
    "utf8_round_trip": arabic_sample.encode("utf-8").decode("utf-8") == arabic_sample,
    "numpy_available": report["packages"]["numpy"] != "not-installed",
    "pandas_available": report["packages"]["pandas"] != "not-installed",
    "sklearn_available": report["packages"]["scikit-learn"] != "not-installed",
    "torch_available": report["packages"]["torch"] != "not-installed",
    "disk_has_1_gib_free": report["disk_free_gib"] >= 1.0,
}

for name, passed in core_checks.items():
    print(("✅" if passed else "❌"), name)

BAYAN_ENV_READY = all(core_checks.values())
print("\nBAYAN_ENV_READY =", BAYAN_ENV_READY)
if not BAYAN_ENV_READY:
    raise RuntimeError("Environment check failed. Copy the failed check names and use the setup troubleshooting guide.")


✅ python_3_10_or_newer
✅ utf8_round_trip
✅ numpy_available
✅ pandas_available
✅ sklearn_available
✅ torch_available
✅ disk_has_1_gib_free

BAYAN_ENV_READY = True


In [3]:
# 🟣 مشروعي — ربط الدفتر بمستودعي: نسخة الكود والبيانات وبصماتها (SHA-256)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi"
_candidates = [Path.cwd(), Path.cwd().parent, Path("/content/bayan-nlp-Manal-qaysi")]
REPO_DIR = next((p for p in _candidates if (p / "src" / "bayan" / "project.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/bayan-nlp-Manal-qaysi") if Path("/content").is_dir() else Path("bayan-nlp-Manal-qaysi").resolve()
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from bayan.project import data_hashes, emit_result, git_head

DATA_DIR = REPO_DIR / "data" / "sample"
REPO_SHA = git_head(REPO_DIR)
DATA_SHA256 = data_hashes(DATA_DIR)
PROVENANCE = {"repo": REPO_URL, "repo_sha": REPO_SHA}
print("REPO_DIR:", REPO_DIR)
print("REPO_SHA:", REPO_SHA)
for _name, _digest in DATA_SHA256.items():
    print(f"  {_name}  sha256={_digest[:16]}…")

REPO_DIR: /content/bayan-nlp-Manal-qaysi
REPO_SHA: 8fabb2a6026a6154d8835bb347e35cce3afeef6e
  bayan_day1_sample.csv  sha256=904a5e1e860f23ac…
  bayan_day2_classification.csv  sha256=c50de92fdab1aa36…
  bayan_day2_ner.jsonl  sha256=ab413f0941656abf…
  bayan_day2_qa.json  sha256=4e894757b74d09df…
  bayan_day3_arabic.csv  sha256=0a3346b6177d0c0b…
  bayan_day3_cases.csv  sha256=322867b54d1f6f35…
  bayan_day3_predictions.csv  sha256=63b4df9dab076880…
  bayan_day3_queries.jsonl  sha256=f80ebd8b25c37b33…


In [4]:
# Connectivity checks are warnings because classroom networks may filter sites.
# Test the exact services used by the notebooks, not only their home pages.
import requests

sites = {
    "github_raw": "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/README.md",
    "pypi": "https://pypi.org/simple/",
    "huggingface_models": "https://huggingface.co/api/models/google-bert/bert-base-multilingual-cased",
}
connectivity = {}
for name, url in sites.items():
    try:
        response = requests.get(
            url, headers={"User-Agent": "Bayan-Course-Runtime-Doctor/1.1"},
            timeout=15, stream=True,
        )
        connectivity[name] = 200 <= response.status_code < 400
        response.close()
    except requests.RequestException as exc:
        connectivity[name] = False
        print(f"⚠️ {name}: {type(exc).__name__}")

report["connectivity"] = connectivity
for name, passed in connectivity.items():
    print(("✅" if passed else "⚠️"), name)

print("\nA connectivity warning does not change BAYAN_ENV_READY.")


✅ github_raw
✅ pypi
✅ huggingface_models

A connectivity warning does not change BAYAN_ENV_READY.


In [5]:
# Save a small, safe report. It contains no password, token, email, or Drive path.
report["core_checks"] = core_checks
report["bayan_env_ready"] = BAYAN_ENV_READY
report_path = Path("runtime_report.json")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", report_path.name)
print("Size:", report_path.stat().st_size, "bytes")


Saved: runtime_report.json
Size: 764 bytes


In [6]:
# 🟣 مشروعي — تسجيل بيئة التشغيل الفعلية (تُستخدم في عقد إعادة الإنتاج في BENCHMARKS.md)
emit_result("nb00_runtime", {
    **PROVENANCE,
    "checked_at_utc": report["checked_at_utc"],
    "in_colab": report["in_colab"],
    "python": report["python"],
    "platform": report["platform"],
    "device": report["device"],
    "gpu_name": report["gpu_name"],
    "packages": report["packages"],
    "bayan_env_ready": BAYAN_ENV_READY,
    "connectivity": report.get("connectivity", {}),
})

BAYAN_RESULT::nb00_runtime::{"bayan_env_ready":true,"checked_at_utc":"2026-09-30T13:36:05.410912+00:00","connectivity":{"github_raw":true,"huggingface_models":true,"pypi":true},"device":"cpu","gpu_name":null,"in_colab":true,"packages":{"numpy":"2.1.3","pandas":"2.2.3","scikit-learn":"1.6.1","torch":"2.11.0+cpu"},"platform":"Linux-6.6.122+-x86_64-with-glibc2.39","python":"3.13.15","repo":"https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi","repo_sha":"8fabb2a6026a6154d8835bb347e35cce3afeef6e"}


'BAYAN_RESULT::nb00_runtime::{"bayan_env_ready":true,"checked_at_utc":"2026-09-30T13:36:05.410912+00:00","connectivity":{"github_raw":true,"huggingface_models":true,"pypi":true},"device":"cpu","gpu_name":null,"in_colab":true,"packages":{"numpy":"2.1.3","pandas":"2.2.3","scikit-learn":"1.6.1","torch":"2.11.0+cpu"},"platform":"Linux-6.6.122+-x86_64-with-glibc2.39","python":"3.13.15","repo":"https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi","repo_sha":"8fabb2a6026a6154d8835bb347e35cce3afeef6e"}'

## اختياري: Google Drive | Optional

لا تشغّل الخلية التالية إلا بعد قراءة صلاحية الوصول. تركيب Drive يسمح للكود بالوصول إلى ملفات Drive. لا تمنح الصلاحية لدفتر غير موثوق.


In [7]:
MOUNT_DRIVE = False  # Change to True only when you choose to mount Drive.

if MOUNT_DRIVE:
    if not in_colab:
        raise RuntimeError("Drive mounting is available in Google Colab, not this runtime.")
    from google.colab import drive
    drive.mount("/content/drive")
    recovery_dir = Path("/content/drive/MyDrive/bayan-nlp/recovery")
    recovery_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(report_path, recovery_dir / report_path.name)
    print("Copied report to:", recovery_dir)
else:
    print("Drive not mounted. This is normal for the first readiness check.")


Drive not mounted. This is normal for the first readiness check.


## النتيجة التالية | Next step

إذا ظهر `BAYAN_ENV_READY = True`:

1. احفظ نسخة notebook في Drive.
2. احتفظ بملف `runtime_report.json`.
3. أكمل [إعداد GitHub](../docs/setup/github.md).
4. لا تقلق إن كان الجهاز CPU؛ هذا يحقق شرط الجاهزية.

If readiness is false, copy the failed check names and open the [troubleshooting guide](../docs/setup/troubleshooting.md).
